<a href="https://colab.research.google.com/github/Asif-Ahmed-Rezvi/flyrank-internship-ml/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Asif-Ahmed-Rezvi/flyrank-internship-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## 1. Signal checks and my rule

### Signal check A — volume (FlyRank quick-win-linked): **CONFIRMED**
The quick-win idea needs enough search demand for a fix to matter. I will test whether higher-impression buckets carry materially more observed clicks. This is a current-window sanity check only; it does not use a target or future window.

### Signal check B — CTR versus position (FlyRank CTR-fix-linked): **CONFIRMED**
The CTR-fix idea needs pages that are already visible but still have low CTR. I will test the share with `ctr < 0.5%` among pages with at least 500 impressions and position 1–20, split into position buckets. This threshold is transparent and uses only the current observed window.

### What I will *not* use
I will not use `trend_pct`, `trend_direction`, `is_declining_label`, any future window, or any product flag as an input. The product flags are background context only.

### Rule in plain words
**Prioritize pages with enough search demand that already rank in the top 20 but have low CTR.** Higher impression volume raises the score, while the CTR gap below 0.5% raises it further.

**Reason code:** `high_visibility_low_ctr` for selected pages; `not_selected` otherwise.

**Action label:** `refresh_and_review_ctr` for selected pages; `monitor` otherwise.

The negative signal check matters: I considered staleness, but the current data do not support treating extreme staleness as a simple positive signal, so staleness is deliberately excluded from this baseline.


In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from pathlib import Path
import os
import numpy as np
import pandas as pd
from IPython.display import display

# Resolve the repository explicitly. The old parents[1] approach fails when
# Colab starts the notebook from the filesystem root instead of the repo.
REPO_DIR = Path("/content/flyrank-internship-ml")
if os.environ.get("COLAB_RELEASE_TAG"):
    if not REPO_DIR.exists():
        !git clone -q https://github.com/Asif-Ahmed-Rezvi/flyrank-internship-ml {REPO_DIR}
    ROOT = REPO_DIR
else:
    candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    ROOT = next(
        (p for p in candidates if (p / "data" / "raw" / "content_refresh_anonymized.csv").exists()),
        None,
    )
    if ROOT is None:
        raise FileNotFoundError(
            "Could not locate the FlyRank repository. Run this notebook from the repository "
            "root, or open it through the provided Colab link."
        )

DATA_PATH = ROOT / "data" / "raw" / "content_refresh_anonymized.csv"
df = pd.read_csv(DATA_PATH)
print(f"Repository: {ROOT}")
print(f"Dataset loaded: {len(df):,} rows × {len(df.columns):,} columns")

# Signal A: volume. Use observed current-window demand only.
volume_bins = [0, 100, 500, 3000, np.inf]
volume_labels = ["1-100", "101-500", "501-3000", "3000+"]
volume_df = df.loc[df["impressions_90d"] > 0].copy()
volume_df["volume_bucket"] = pd.cut(
    volume_df["impressions_90d"],
    bins=volume_bins,
    labels=volume_labels,
    include_lowest=True,
)
volume_table = (
    volume_df.groupby("volume_bucket", observed=False)
    .agg(
        n=("content_id", "size"),
        median_impressions=("impressions_90d", "median"),
        median_clicks=("clicks_90d", "median"),
        mean_clicks=("clicks_90d", "mean"),
    )
    .reset_index()
)
print("Signal A — volume bucket table")
display(volume_table)

# Signal B: CTR versus position, with a volume floor.
ctr_df = df.loc[
    (df["impressions_90d"] >= 500)
    & (df["avg_position"] > 0)
    & (df["avg_position"] <= 20)
].copy()
ctr_df["position_bucket"] = pd.cut(
    ctr_df["avg_position"],
    bins=[0, 3, 10, 20],
    labels=["top_3", "page_1", "positions_11-20"],
    include_lowest=True,
)
ctr_table = (
    ctr_df.groupby("position_bucket", observed=False)
    .agg(
        n=("content_id", "size"),
        median_ctr_pct=("ctr", "median"),
        low_ctr_share_pct=("ctr", lambda s: 100 * (s < 0.5).mean()),
        median_impressions=("impressions_90d", "median"),
    )
    .reset_index()
)
print("Signal B — CTR versus position bucket table")
display(ctr_table)

print(
    f"Verdicts: volume=CONFIRMED; CTR-vs-position=CONFIRMED. "
    f"CTR-vs-position test uses n={len(ctr_df):,} visible, volume-qualified pages."
)


Repository: /content/flyrank-internship-ml
Dataset loaded: 30,000 rows × 44 columns
Signal A — volume bucket table


,volume_bucket,n,median_impressions,median_clicks,mean_clicks
0,1-100,8006,12.0,0.0,0.146765
1,101-500,5279,253.0,0.0,0.586475
2,501-3000,8432,1231.0,2.0,3.207780
3,3000+,8283,8426.0,20.0,54.521429


Signal B — CTR versus position bucket table


,position_bucket,n,median_ctr_pct,low_ctr_share_pct,median_impressions
0,top_3,480,0.20,75.000000,4382.5
1,page_1,7084,0.24,79.178430,4502.0
2,positions_11-20,4459,0.17,84.996636,2166.0


Verdicts: volume=CONFIRMED; CTR-vs-position=CONFIRMED. CTR-vs-position test uses n=12,023 visible, volume-qualified pages.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

The score is intentionally small and readable:

`score = log1p(impressions_90d) × CTR-gap × visibility gate`

where:
- `CTR-gap = max(0, 0.5 - ctr) / 0.5`
- visibility requires `impressions_90d >= 500` and `0 < avg_position <= 20`
- the action is `refresh_and_review_ctr` when the score is positive, otherwise `monitor`
- each row gets exactly one reason code.

This is a ranking baseline, not a prediction of future performance.


In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

# Transparent, non-fitted rule.
volume_gate = (df["impressions_90d"] >= 500).astype(int)
position_gate = ((df["avg_position"] > 0) & (df["avg_position"] <= 20)).astype(int)
ctr_gap = np.clip(0.5 - df["ctr"], 0, 0.5) / 0.5

queue = df[
    [
        "content_id",
        "client_id",
        "impressions_90d",
        "clicks_90d",
        "ctr",
        "avg_position",
        "days_since_last_update",
        "content_age_days",
    ]
].copy()

queue["score"] = (
    np.log1p(queue["impressions_90d"])
    * ctr_gap
    * volume_gate
    * position_gate
)
queue["reason_code"] = np.where(
    queue["score"] > 0,
    "high_visibility_low_ctr",
    "not_selected",
)
queue["action"] = np.where(
    queue["score"] > 0,
    "refresh_and_review_ctr",
    "monitor",
)

queue = queue.sort_values(
    ["score", "impressions_90d", "ctr"],
    ascending=[False, False, True],
    kind="mergesort",
).reset_index(drop=True)
queue["rank"] = np.arange(1, len(queue) + 1)

# Keep the ranked queue useful but avoid exposing client identifiers in the artifact.
output_columns = [
    "rank",
    "content_id",
    "score",
    "reason_code",
    "action",
    "impressions_90d",
    "ctr",
    "avg_position",
]
output_path = ROOT / "work" / "outputs" / "baseline_action_score.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)
queue[output_columns].to_csv(output_path, index=False)

# Save a small run receipt; the CSV itself is intentionally git-ignored.
metrics = {
    "rows_ranked": int(len(queue)),
    "selected_for_ctr_review": int((queue["score"] > 0).sum()),
    "max_score": float(queue["score"].max()),
    "signal_verdicts": {
        "volume": "CONFIRMED",
        "ctr_vs_position": "CONFIRMED",
    },
    "rule_inputs": ["impressions_90d", "ctr", "avg_position"],
    "future_or_label_inputs": [],
}
metrics_path = ROOT / "work" / "outputs" / "w04_baseline_metrics.json"
metrics_path.write_text(__import__("json").dumps(metrics, indent=2))
print(f"Wrote run receipt to {metrics_path}")

print(f"Wrote {len(queue):,} ranked rows to {output_path}")
print(f"Selected for CTR review: {(queue['score'] > 0).sum():,} rows")
print(f"Score range: {queue['score'].min():.3f} to {queue['score'].max():.3f}")
display(queue.head(10)[output_columns])

Wrote run receipt to /content/flyrank-internship-ml/work/outputs/w04_baseline_metrics.json
Wrote 30,000 ranked rows to /content/flyrank-internship-ml/work/outputs/baseline_action_score.csv
Selected for CTR review: 9,759 rows
Score range: 0.000 to 12.249


,rank,content_id,score,reason_code,action,impressions_90d,ctr,avg_position
0,1,content_c8e9d6ab9013,12.248552,high_visibility_low_ctr,refresh_and_review_ctr,208678,0.00,9.7
1,2,content_8451fc6f034d,11.763245,high_visibility_low_ctr,refresh_and_review_ctr,272144,0.03,2.3
2,3,content_453722754fea,11.612970,high_visibility_low_ctr,refresh_and_review_ctr,140079,0.01,7.6
3,4,content_c84a0ab98e90,11.577177,high_visibility_low_ctr,refresh_and_review_ctr,223271,0.03,7.8
4,5,content_4a6607efcb46,11.525118,high_visibility_low_ctr,refresh_and_review_ctr,128068,0.01,2.2
5,6,content_39881853ef0c,11.397528,high_visibility_low_ctr,refresh_and_review_ctr,112434,0.01,7.2
6,7,content_36ff89c8214e,11.335557,high_visibility_low_ctr,refresh_and_review_ctr,295097,0.05,7.3
7,8,content_0919dd345d80,11.221161,high_visibility_low_ctr,refresh_and_review_ctr,119217,0.02,7.0
8,9,content_c1fe78bc4e37,11.097652,high_visibility_low_ctr,refresh_and_review_ctr,134055,0.03,7.5
9,10,content_b115f7c74779,11.020328,high_visibility_low_ctr,refresh_and_review_ctr,123469,0.03,8.0


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Each line states the action, why the page is ranked, and one concrete observation that could make the rule wrong. The review uses only fields available at the decision point.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Create the required one-line review for the top ten.
top10 = queue.head(10).copy()

def wrong_if(row):
    if row["ctr"] == 0:
        return "a measurement or attribution issue is making CTR appear artificially low"
    if row["avg_position"] <= 3:
        return "the query mix is largely informational/zero-click or SERP features suppress clicks despite a strong rank"
    if row["impressions_90d"] > 200_000:
        return "the high impression total comes from broad or low-intent queries where a CTR change would not be a useful content action"
    return "the position average hides weaker query-level rankings or the observed CTR is not an actionable content problem"

reviews = []
for _, r in top10.iterrows():
    reviews.append(
        f"#{int(r['rank'])} — action={r['action']}; "
        f"why={r['impressions_90d']:,.0f} impressions, CTR={r['ctr']:.2f}% at "
        f"position {r['avg_position']:.1f}, so it has high visibility with a measurable CTR gap; "
        f"wrong if={wrong_if(r)}."
    )

for line in reviews:
    print(line)


#1 — action=refresh_and_review_ctr; why=208,678 impressions, CTR=0.00% at position 9.7, so it has high visibility with a measurable CTR gap; wrong if=a measurement or attribution issue is making CTR appear artificially low.
#2 — action=refresh_and_review_ctr; why=272,144 impressions, CTR=0.03% at position 2.3, so it has high visibility with a measurable CTR gap; wrong if=the query mix is largely informational/zero-click or SERP features suppress clicks despite a strong rank.
#3 — action=refresh_and_review_ctr; why=140,079 impressions, CTR=0.01% at position 7.6, so it has high visibility with a measurable CTR gap; wrong if=the position average hides weaker query-level rankings or the observed CTR is not an actionable content problem.
#4 — action=refresh_and_review_ctr; why=223,271 impressions, CTR=0.03% at position 7.8, so it has high visibility with a measurable CTR gap; wrong if=the high impression total comes from broad or low-intent queries where a CTR change would not be a useful c

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

The main weakness is conceptual rather than computational: low CTR is not automatically a content defect. SERP features, query intent, brand demand, and measurement quality can all lower observed CTR. I therefore treat the queue as review candidates, not automatic fixes.

The earlier staleness check was deliberately rejected as a rule input because extreme staleness was not monotonic with current demand in this slice. That is a useful negative finding: it prevented the baseline from rewarding stale pages simply because they are old.

The leakage check below verifies that the rule and queue do not use target/future fields or product decision flags.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Show a few weak candidates from the selected set.
selected = queue.loc[queue["score"] > 0].copy()
weak = selected.sort_values(
    ["score", "impressions_90d"],
    ascending=[True, True],
).head(5)

print("Weak-pick examples: these are the lowest-scoring selected pages, so they are the first candidates to challenge in review.")
display(
    weak[
        ["rank", "content_id", "score", "reason_code", "action",
         "impressions_90d", "ctr", "avg_position"]
    ]
)

print(
    "Weak-pick lesson: the rule can surface a page with low CTR because the observed "
    "signals fit the threshold even when the underlying cause is not a content problem."
)

Weak-pick examples: these are the lowest-scoring selected pages, so they are the first candidates to challenge in review.


,rank,content_id,score,reason_code,action,impressions_90d,ctr,avg_position
9758,9759,content_05dc29232a47,0.128269,high_visibility_low_ctr,refresh_and_review_ctr,609,0.49,4.1
9757,9758,content_c60095baf03c,0.128335,high_visibility_low_ctr,refresh_and_review_ctr,611,0.49,5.1
9756,9757,content_c51d82ffafda,0.128530,high_visibility_low_ctr,refresh_and_review_ctr,617,0.49,12.0
9755,9756,content_f51e681dcb3e,0.134015,high_visibility_low_ctr,refresh_and_review_ctr,812,0.49,7.5
9754,9755,content_83f1c6044e8d,0.134064,high_visibility_low_ctr,refresh_and_review_ctr,814,0.49,6.6


Weak-pick lesson: the rule can surface a page with low CTR because the observed signals fit the threshold even when the underlying cause is not a content problem.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.